# Cleaning the UN Canada Immigration Dataset (1980–2013)

**Assignment goal:** reshape the United Nations immigration spreadsheet into a country-indexed DataFrame with four descriptive fields, annual immigration counts, and a `Total` column.

The target layout has `Country` as the index, followed by `Continent`, `Region`, `DevName`, the years 1980–2013, and `Total`.

> **Data note:** The Google Sheets link supplied for this assignment is used as the live source. The notebook includes a small, clearly labelled preview fixture matching the visible example rows so that the transformation can be demonstrated when the source spreadsheet is not reachable. Re-run the source-loading cell in an environment with access to the spreadsheet to process the complete dataset.


## 1. Import the required libraries

We use pandas for reading and transforming tabular data. `display` gives a readable notebook preview.


In [1]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)


## 2. Read the source spreadsheet

The Google Sheets export endpoint returns an Excel workbook. The original UN workbook layout has explanatory material above the data table, so we skip the first 20 rows. The `sheet_name=0` setting selects the first worksheet; change it if your workbook uses a different sheet order.

If access is restricted or the network is unavailable, the cell reports the issue and the following cells use a small preview fixture instead. The fixture is not a replacement for the full UN dataset.


In [2]:
# Google Sheets workbook supplied in the assignment.
SHEET_ID = "1OGs8KoHGEZDxyFdaxAKmZbeO1BiSWVQ6"
SOURCE_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx"

raw_df = None
source_loaded = False

try:
    # Read the data table after the UN report's introductory rows.
    raw_df = pd.read_excel(
        SOURCE_URL,
        sheet_name=0,
        skiprows=range(20),
        skipfooter=2,
        engine="openpyxl"
    )
    source_loaded = True
    print(f"Source spreadsheet loaded successfully: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns")
    display(raw_df.head())
except Exception as error:
    print("The Google Sheet could not be reached from this environment.")
    print(f"Details: {type(error).__name__}: {error}")
    print("A small preview fixture will be used below; rerun this cell where the sheet is accessible.")


The Google Sheet could not be reached from this environment.
Details: URLError: <urlopen error [Errno -3] Temporary failure in name resolution>
A small preview fixture will be used below; rerun this cell where the sheet is accessible.


## 3. Prepare a small preview fixture (only if needed)

The target screenshot visibly shows the first five countries and selected years (the middle years are abbreviated with an ellipsis). This fixture records those visible values only. It is used solely to demonstrate the layout when the live workbook cannot be loaded; it does **not** fabricate the omitted years or claim to be the complete dataset.


In [3]:
if not source_loaded:
    # Values transcribed from the visible target screenshot.
    # Only years that are legible in the screenshot are included in this preview.
    preview_data = {
        "OdName": ["Afghanistan", "Albania", "Algeria", "American Samoa", "Andorra"],
        "AreaName": ["Asia", "Europe", "Africa", "Oceania", "Europe"],
        "RegName": ["Southern Asia", "Southern Europe", "Northern Africa", "Polynesia", "Southern Europe"],
        "DevName": ["Developing regions", "Developed regions", "Developing regions", "Developing regions", "Developed regions"],
        1980: [16, 1, 80, 0, 0],
        1981: [39, 0, 67, 1, 0],
        1982: [39, 0, 71, 0, 0],
        1983: [47, 0, 69, 0, 0],
        1984: [71, 0, 63, 0, 0],
        1985: [340, 0, 44, 0, 0],
        1986: [496, 1, 69, 0, 2],
        2005: [3436, 1223, 3626, 0, 0],
        2006: [3009, 856, 4807, 1, 1],
        2007: [2652, 702, 3623, 0, 1],
        2008: [2111, 560, 4005, 0, 0],
        2009: [1746, 716, 5393, 0, 0],
        2010: [1758, 561, 4752, 0, 0],
        2011: [2203, 539, 4325, 0, 0],
        2012: [2635, 620, 3774, 0, 1],
        2013: [2004, 603, 4331, 0, 1],
        "Total": [58639, 15699, 69439, 6, 15],
    }
    raw_df = pd.DataFrame(preview_data)

display(raw_df.head())


,OdName,AreaName,RegName,DevName,1980,1981,1982,1983,1984,1985,1986,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
0,Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
1,Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,1223,856,702,560,716,561,539,620,603,15699
2,Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
3,American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,6
4,Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,0,1,1,0,0,0,0,1,1,15


## 4. Clean and reshape the DataFrame

The transformation:
1. Removes technical classification-code columns and source metadata (`Type`, `Coverage`).
2. Renames the country, continent, and region fields to readable labels.
3. Places the country names in the index.
4. Converts year labels to strings so the columns are consistent and easy to select.
5. Adds `Total` if the source does not already provide it.
6. Orders the descriptive fields first, annual values next, and `Total` last.

The code detects year columns from their labels, so it works with all years in the complete source as well as the abbreviated preview.


In [4]:
def make_target_dataframe(source):
    """Return the immigration table in the requested country-indexed layout."""
    df = source.copy()

    # Remove fields that are not part of the requested final table.
    columns_to_drop = ["AREA", "REG", "DEV", "Type", "Coverage"]
    df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])

    # Rename source headings to the labels shown in the assignment.
    rename_map = {
        "OdName": "Country",
        "AreaName": "Continent",
        "RegName": "Region",
    }
    df = df.rename(columns=rename_map)

    # Identify year columns, regardless of whether Excel read them as numbers or strings.
    year_columns = [
        col for col in df.columns
        if str(col).strip().isdigit() and 1980 <= int(str(col).strip()) <= 2013
    ]
    year_columns = sorted(year_columns, key=lambda col: int(str(col).strip()))

    # Convert annual values to numeric values for reliable totals.
    for col in year_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Calculate Total only if the source does not already contain a Total column.
    if "Total" not in df.columns:
        df["Total"] = df[year_columns].sum(axis=1)

    # Put the country in the index and arrange columns in the requested order.
    df = df.set_index("Country")
    df.columns = [str(col) for col in df.columns]

    year_columns = [str(col) for col in year_columns]
    descriptive_columns = ["Continent", "Region", "DevName"]
    ordered_columns = descriptive_columns + year_columns + ["Total"]
    ordered_columns = [col for col in ordered_columns if col in df.columns]

    df = df.loc[:, ordered_columns].sort_index()
    df.index.name = "Country"

    return df


immigration_df = make_target_dataframe(raw_df)
print("Transformed DataFrame:")
display(immigration_df.head())


Transformed DataFrame:


,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,0,1,1,0,0,0,0,1,1,15


## 5. Validate the result

These checks confirm the expected index, descriptive columns, year range (when the complete source is available), and final `Total` column. In preview mode, only the years visible in the screenshot are present.


In [5]:
# Basic structural checks.
assert immigration_df.index.name == "Country"
assert immigration_df.index.is_unique
assert ["Continent", "Region", "DevName"] == [
    col for col in immigration_df.columns if col in ["Continent", "Region", "DevName"]
]
assert immigration_df.columns[-1] == "Total"

print(f"Rows (countries): {len(immigration_df)}")
print(f"Columns: {len(immigration_df.columns)}")
print(f"First country: {immigration_df.index[0]}")
print(f"Last column: {immigration_df.columns[-1]}")

if source_loaded:
    expected_years = [str(year) for year in range(1980, 2014)]
    missing_years = [year for year in expected_years if year not in immigration_df.columns]
    assert not missing_years, f"Missing year columns: {missing_years}"
    print("All annual columns from 1980 through 2013 are present.")
else:
    print("Preview mode: this is not the complete 1980–2013 dataset.")


Rows (countries): 5
Columns: 20
First country: Afghanistan
Last column: Total
Preview mode: this is not the complete 1980–2013 dataset.


## 6. Export the cleaned table

The cleaned DataFrame can be saved as a CSV file for later use. The country index is retained in the output file.


In [6]:
OUTPUT_FILE = "canada_immigration_cleaned.csv"
immigration_df.to_csv(OUTPUT_FILE, index=True)
print(f"Saved cleaned data to: {OUTPUT_FILE}")


Saved cleaned data to: canada_immigration_cleaned.csv
